In [ ]:
# SerpAPI-based showtimes fetch and diagnostic cell
# Use this cell to call SerpAPI (engine='google') and extract time-like strings from the JSON response.

# Use this cell to call SerpAPI (engine='google') and extract time-like strings from the JSON response.

import os
import requests
import re
import json
from typing import Tuple, Dict, Any, Optional

TIME_RE = re.compile(r"\b\d{1,2}[:h]\d{2}\s*(?:[ap]m)?\b", re.I)

def get_showtimes_serpapi(cinema: str, location: Optional[str] = None, api_key: Optional[str] = None, timeout: int = 30) -> Tuple[Dict[str, Any], Dict[str, list]]:
    """Query SerpAPI using engine='google' and return (raw_json, matches).

    - Saves `serpapi_showtimes.json` in the repo for inspection.
    - Returns a mapping of JSON-path -> list of time-like strings found there.
    """
    if api_key is None:
        api_key = os.getenv('SERP_API')
    if not api_key:
        raise RuntimeError('SERP_API environment variable is not set')

    q = f"showtimes {cinema} {location or ''}".strip()
    params = {
        'engine': 'google',
        'q': q,
        'hl': 'en',
        'api_key': api_key,
    }
    if location:
        params['location'] = location

    try:
        r = requests.get('https://serpapi.com/search.json', params=params, timeout=timeout)
        r.raise_for_status()
    except requests.HTTPError as http_err:
        status = getattr(http_err.response, 'status_code', 'N/A') if hasattr(http_err, 'response') else 'N/A'
        body = http_err.response.text if (hasattr(http_err, 'response') and http_err.response is not None) else ''
        print(f"SerpAPI request failed: {http_err}\nStatus code: {status}\nResponse body (truncated): {body[:1000]!r}")
        raise
    except Exception as e:
        print('Failed to contact SerpAPI:', e)
        raise

    data = r.json()
    # Save raw response for inspection
    try:
        with open('serpapi_showtimes.json', 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print('Saved SerpAPI raw response to serpapi_showtimes.json')
    except Exception as e:
        print('Failed to write serpapi_showtimes.json:', e)

    matches: dict[str, list] = {}

    def traverse(obj, path='root'):
        if isinstance(obj, str):
            found = TIME_RE.findall(obj)
            if found:
                matches[path] = found
        elif isinstance(obj, dict):
            for k, v in obj.items():
                traverse(v, f"{path}/{k}")
        elif isinstance(obj, list):
            for i, v in enumerate(obj):
                traverse(v, f"{path}[{i}]")

    traverse(data, 'root')

    print('Top-level keys in SerpAPI response:', list(data.keys()))
    print(f'Found {len(matches)} JSON paths containing time-like strings (sample up to 30):')
    import pprint
    pprint.pprint({k: matches[k] for k in list(matches)[:30]})

    return data, matches

# Example usage (adjust cinema/location as needed)
try:
    cinema = "L'Arlequin"
    location = "Paris, France"
    data, matches = get_showtimes_serpapi(cinema, location)
source
import os, json, html
from typing import Any, Dict, List

def load_serpapi_data() -> Dict[str, Any]:
    try:
        d = globals().get('data')
        if isinstance(d, dict):
            return d
    except Exception:
        pass
    fname = 'serpapi_showtimes.json'
    if os.path.exists(fname):
        with open(fname, 'r', encoding='utf-8') as f:
            return json.load(f)
    raise RuntimeError('No SerpAPI data found. Run get_showtimes_serpapi(...) first.')

def render_html_table(rows: List[Dict[str, Any]]) -> str:
    style = '''<style>













































    print('Failed to build showtimes table:', e)except Exception as e:    display(HTML(html_out))    from IPython.display import HTML, display    print(f'Wrote {out_file} (open in browser to view)')        f.write(html_out)    with open(out_file, 'w', encoding='utf-8') as f:    out_file = 'showtimes_table.html'    html_out = render_html_table(candidates)    print(f'Found {len(candidates)} film candidates')            candidates.append({'title': path, 'director': None, 'year': None, 'country': None, 'cinema': None, 'schedule': {'default': times}})        for path, times in list(matches.items())[:200]:        candidates = []        matches = globals().get('matches') or {}        print('extract_candidates_from_json not found; using fallback from matches')    else:        candidates = extract_candidates_from_json(data)    if 'extract_candidates_from_json' in globals():    data = load_serpapi_data()try:# Build candidates either via the main extractor or a simple fallback    return '\n'.join(parts)    parts.append('</body></html>')    parts.append('</tbody></table>')        parts.append(f'<tr><td>{title}</td><td>{director}</td><td>{year}</td><td>{country}</td><td>{cinema}</td><td>{times_html}</td></tr>')        times_html = '<br>'.join(day_lines)            day_lines.append(f"{html.escape(str(day))}: {html.escape(times_str)}")            times_str = ', '.join(times)                continue            if not times:        for day, times in schedule.items():        day_lines = []        schedule = r.get('schedule') or {}        cinema = html.escape(r.get('cinema') or '')        country = html.escape(r.get('country') or '')        year = html.escape(r.get('year') or '')        director = html.escape(r.get('director') or '')        title = html.escape(r.get('title') or '')    for r in rows:    parts.append('<tbody>')    parts.append('<thead><tr><th>Title</th><th>Director</th><th>Year</th><th>Country</th><th>Cinema</th><th>Days & Times</th></tr></thead>')    parts.append('<table>')    parts = ["<html><head>", style, "</head><body>"]    table {border-collapse: collapse; width: 100%; font-family: Arial, sans-serif}
    th, td {border: 1px solid #ccc; padding: 6px; text-align: left}
    th {background: #f3f3f3}
    </style>'''

Saved SerpAPI raw response to serpapi_showtimes.json
Top-level keys in SerpAPI response: ['search_metadata', 'search_parameters', 'search_information', 'showtimes', 'knowledge_graph', 'related_questions', 'ai_overview', 'organic_results', 'menu_highlights', 'related_searches', 'pagination', 'serpapi_pagination']
Found 40 JSON paths containing time-like strings (sample up to 30):
{'root/search_metadata/created_at': ['15:53'],
 'root/search_metadata/processed_at': ['15:53'],
 'root/showtimes[0]/movies[0]/showing[0]/time[0]': ['5:10pm'],
 'root/showtimes[0]/movies[0]/showing[0]/time[1]': ['7:15pm'],
 'root/showtimes[0]/movies[0]/showing[0]/time[2]': ['9:20pm'],
 'root/showtimes[0]/movies[1]/showing[0]/time[0]': ['5:00pm'],
 'root/showtimes[0]/movies[1]/showing[0]/time[1]': ['8:15pm'],
 'root/showtimes[0]/movies[2]/showing[0]/time[0]': ['6:30pm'],
 'root/showtimes[0]/movies[3]/showing[0]/time[0]': ['8:50pm'],
 'root/showtimes[1]/movies[0]/showing[0]/time[0]': ['1:00pm'],
 'root/showtimes[1